# 运行取消

运行取消是 Agent Server 独有的功能。

当用户不再需要结果，或者任务执行时间过长时，可以通过 `cancel` 接口停止运行。

In [ ]:
from langgraph_sdk import get_client

client = get_client(url="http://localhost:2024")

## 准备演示

本课使用 `cancel_demo_graph`。这个图中有几个慢节点，完整运行大约需要 6 秒，方便我们在它结束前取消。

In [ ]:
assistant_id = "efbf07f8-c28f-4db8-a7ff-17b58c4af012"

# 删除所有线程，免得看晕了
threads = await client.threads.search(limit=100)
for t in threads:
    await client.threads.delete(thread_id=t['thread_id'])

thread = await client.threads.create(
    metadata={"__name__": "运行取消演示"}
)
thread_id = thread["thread_id"]
thread_id

## 创建后台运行

`client.runs.create()` 创建 Run 后会立即返回，图在 Agent Server 后台继续执行。

我们需要保留 `run_id`，因为取消接口需要同时知道 `thread_id` 和 `run_id`。

In [ ]:
run = await client.runs.create(
    thread_id=thread_id,
    assistant_id=assistant_id,
    input={
        "messages": [
            {"role":"user", "content":"你好！"}
        ]
    },
)
run_id = run["run_id"]
run["status"]

In [ ]:
cur_run = await client.runs.get(thread_id, run_id)
cur_run["status"]

## 取消运行

调用 `client.runs.cancel()` 取消运行：

- `action="interrupt"`：停止运行，但保留 Run 记录和已经产生的检查点
- `wait=True`：等到取消真正完成后再返回

`interrupt` 是默认的 action，这里为了讲解清楚，将它显式写出来。

In [ ]:
await client.runs.cancel(
    thread_id=thread_id,
    run_id=run_id,
    action="interrupt", # interrupt是默认值，另一个值是 rollback
    wait=True,
)

print("取消请求已完成")